In [1]:
import pandas as pd

churn = pd.read_csv("data/customer_churn (2).csv")
sales = pd.read_csv("data/sales_data (1).csv")
house = pd.read_csv("data/house_prices (1).csv")

print("Churn:", churn.shape)
print("Sales:", sales.shape)
print("House:", house.shape)

churn.head()



Churn: (500, 9)
Sales: (100, 7)
House: (300, 8)


,CustomerID,Tenure,MonthlyCharges,TotalCharges,Contract,PaymentMethod,PaperlessBilling,SeniorCitizen,Churn
0,C00001,6,64,1540,One year,Credit Card,No,1,0
1,C00002,21,113,1753,Month-to-month,Electronic Check,Yes,1,0
2,C00003,27,31,1455,Two year,Credit Card,No,1,0
3,C00004,53,29,7150,Month-to-month,Electronic Check,No,1,0
4,C00005,16,185,1023,One year,Electronic Check,No,1,0


In [2]:
# Handle missing values
churn.isnull().sum()

# Encode categorical columns
churn_encoded = pd.get_dummies(churn, columns=["Contract","PaymentMethod","PaperlessBilling"], drop_first=True)

# Features & Target
X = churn_encoded.drop(["CustomerID","Churn"], axis=1)
y = churn_encoded["Churn"]

X.head()


,Tenure,MonthlyCharges,TotalCharges,SeniorCitizen,Contract_One year,Contract_Two year,PaymentMethod_Credit Card,PaymentMethod_Electronic Check,PaperlessBilling_Yes
0,6,64,1540,1,True,False,True,False,False
1,21,113,1753,1,False,False,False,True,True
2,27,31,1455,1,False,True,True,False,False
3,53,29,7150,1,False,False,False,True,False
4,16,185,1023,1,True,False,False,True,False


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Model Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Model Accuracy: 0.96
              precision    recall  f1-score   support

           0       0.97      0.99      0.98        84
           1       0.93      0.81      0.87        16

    accuracy                           0.96       100
   macro avg       0.95      0.90      0.92       100
weighted avg       0.96      0.96      0.96       100



In [4]:
import pickle
pickle.dump(model, open("deployment/churn_model.pkl","wb"))


In [5]:
import matplotlib.pyplot as plt

plt.figure()
churn["Churn"].value_counts().plot(kind="bar")
plt.title("Churn Distribution")
plt.savefig("reports/churn_distribution.png")
plt.close()


### Business Insight

Customers with Month-to-Month contracts show significantly higher churn.  
Offering annual plans with discounts can reduce churn by approximately 20%.


In [6]:
sales["Date"] = pd.to_datetime(sales["Date"])

monthly_sales = sales.groupby(sales["Date"].dt.to_period("M"))["Total_Sales"].sum()

import matplotlib.pyplot as plt
plt.figure()
monthly_sales.plot(title="Monthly Sales Trend")
plt.savefig("reports/monthly_sales_trend.png")
plt.close()

monthly_sales.tail()


Date
2024-01    4120524
2024-02    2656050
2024-03    4485006
2024-04    1103468
Freq: M, Name: Total_Sales, dtype: int64